# Thu thập và chuẩn bị dữ liệu bình luận PTIT

Quy trình gồm thu thập bài viết và bình luận, chuẩn hóa cấu trúc, làm sạch văn
bản, gán nhãn Spam/Ham và xuất tập dữ liệu CSV.


## 1. Cấu hình môi trường

Sử dụng Python 3.12 và các thư viện trong `requirements/notebook.txt`.
Chế độ tái lập sử dụng kết quả thu thập và phản hồi gán nhãn đã lưu;
các thao tác yêu cầu phiên Facebook hoặc API được tổ chức trong module riêng.


In [1]:
import os, sys
from pathlib import Path

# Xác định thư mục gốc để nạp các module của project.
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "pipeline/__init__.py").is_file()), None)
if PROJECT_ROOT is None:
    raise RuntimeError("Mở notebook trong thư mục project có pipeline/ và external/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))



import json, subprocess
subprocess.run([sys.executable, str(PROJECT_ROOT / "scripts/check_environment.py")], check=True)
import pandas as pd
from IPython.display import HTML
from pipeline.collect.json_to_csv import flatten_posts
from pipeline.prepare.clean_for_labeling import clean_facebook_comments
from pipeline.label.cached_annotations import attach_annotation_ids
from pipeline.presentation.notebook_display import (
    display_readable as display, code_excerpt_html, file_record)

SOURCE_ROOT = Path(os.environ.get("SPAMHAM_SOURCE_DIR", str(PROJECT_ROOT / ".cache/collection")))
DATA_DIR = Path(os.environ.get("SPAMHAM_OUTPUT_DIR", str(PROJECT_ROOT / "data")))
INTERMEDIATE_DIR = DATA_DIR / "intermediate"
LABELED_DIR = DATA_DIR / "labeled"
INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
LABELED_DIR.mkdir(parents=True, exist_ok=True)
inputs = {"posts": SOURCE_ROOT / "posts.json",
          "annotation_index": SOURCE_ROOT / "annotation_index.csv",
          "predictions": SOURCE_ROOT / "annotation_predictions.json"}

def show_code(relative_path, function_name, max_lines=30):
    excerpt = code_excerpt_html(PROJECT_ROOT / relative_path, function_name, max_lines, title=relative_path)
    summary = '<summary style="background:#e2e8f0!important;color:#0f172a!important;padding:9px;cursor:pointer;">Mã nguồn: '+function_name+'</summary>'
    display(HTML('<details>'+summary+excerpt+'</details>'))


## 2. Thu thập bài viết và bình luận

Runner `scrape_ptit_groups.py` cấu hình group, giới hạn và mốc thời gian;
`fetch_posts()` lấy bài, `fetch_comments_for_post()` lấy bình luận và phản hồi.
Writer lưu nội dung cùng group ID, post ID, permalink và thời gian vào JSON.

Bảng dưới mô tả phạm vi thu thập và một số nội dung thuộc các nhóm PTIT.


In [2]:
posts = json.loads(inputs["posts"].read_text(encoding="utf-8-sig"))
raw_rows = pd.DataFrame(flatten_posts(posts))
groups = pd.DataFrame([{"Nhóm": p["group_name"], "ID": p["group_id"]} for p in posts])
group_table = groups.groupby(["Nhóm", "ID"]).size().rename("Số bài viết").reset_index()
group_table["Liên kết"] = "https://www.facebook.com/groups/"+group_table.ID.astype(str)+"/"
display(group_table)
display(pd.Series({"Nhóm": len(group_table), "Bài viết": len(posts),
                   "Bình luận": int(raw_rows.comment_type.eq("comment").sum()),
                   "Phản hồi": int(raw_rows.comment_type.eq("reply").sum())}).to_frame("Số lượng"))
example = next(p for p in posts if any(c.get("replies") for c in p.get("comments", [])))
display(pd.Series({key: example.get(key) for key in ["group_name", "post_id", "permalink", "created_at"]}).to_frame("Bài viết"))
example_rows = pd.DataFrame(flatten_posts([example]))
display(pd.concat([example_rows[example_rows.comment_type.eq(kind)].head(3) for kind in ["comment", "reply"]])[
    ["post_id", "comment_type", "text", "reaction_count"]])
show_code("pipeline/collect/scrape_ptit_groups.py", "on_batch")
show_code("external/facebook_scraper/main.py", "fetch_comments_for_post")
show_code("pipeline/collect/scrape_ptit_groups.py", "save_records")


#,Nhóm,ID,Số bài viết,Liên kết
0,Cộng đồng sinh viên PTIT,450310163844833,500,https://www.facebook.com/groups/450310163844833/
1,D21 PTIT Hà Nội,778910719727961,170,https://www.facebook.com/groups/778910719727961/
2,D26 (2008) PTIT - Học viện Công nghệ Bưu chính Viễn thông,1130813927934700,500,https://www.facebook.com/groups/1130813927934700/
3,D26 Học Viện Công Nghệ Bưu Chính Viễn Thông (2K8 PTIT),1605563914144178,500,https://www.facebook.com/groups/1605563914144178/
4,PTIT Confessions - Sinh Viên Học Viện Công Nghệ Bưu Chính Viễn Thông ✅,1210825580901941,500,https://www.facebook.com/groups/1210825580901941/


#,Số lượng
Nhóm,5
Bài viết,2170
Bình luận,9438
Phản hồi,4395


#,Bài viết
group_name,D21 PTIT Hà Nội
post_id,2015992686019752
permalink,https://www.facebook.com/groups/d21ptithanoii/posts/2015992686019752/
created_at,2026-03-25 17:35:43


#,post_id,comment_type,text,reaction_count
0,2015992686019752,comment,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,1
1,2015992686019752,comment,"Cô Chi nhé, bao điểm cao luôn",0
2,2015992686019752,comment,Thầy hùng nhé,0
3,2015992686019752,reply,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,0


## 3. Chuẩn hóa cấu trúc dữ liệu

Chuyển cấu trúc bài viết–bình luận–phản hồi thành bảng, mỗi dòng tương ứng một
bình luận hoặc phản hồi. Giữ metadata nguồn và loại nội dung rỗng.


In [3]:
comments = raw_rows[raw_rows.text.str.strip().ne("")].reset_index(drop=True)
comments_path = INTERMEDIATE_DIR / "comments.csv"
comments.to_csv(comments_path, index=False, encoding="utf-8-sig")
comments = pd.read_csv(comments_path, dtype=str, keep_default_na=False)
display(comments[["post_id", "comment_type", "text", "reaction_count"]].head(8))
display(pd.Series({"Nội dung trích xuất": len(raw_rows), "Nội dung rỗng": len(raw_rows)-len(comments),
                   "Bình luận hợp lệ": len(comments)}).to_frame("Số dòng"))
show_code("pipeline/collect/json_to_csv.py", "flatten_posts")


#,post_id,comment_type,text,reaction_count
0,2015992686019752,comment,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,1
1,2015992686019752,comment,"Cô Chi nhé, bao điểm cao luôn",0
2,2015992686019752,comment,Thầy hùng nhé,0
3,2015992686019752,reply,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,0
4,2015992686019752,comment,"Thầy Kiên nha em, thầy 10 điểm ko có nhưng",0
5,2015992686019752,comment,Cô thư điểm cao,0
6,2179604596325226,comment,Chụp ảnh ib anh tư vấn nhe,0
7,2179604596325226,reply,Nông Thành Long có đưa ảnh em thì dùng cái pts rồi nghe chưa :)))),1


#,Số dòng
Nội dung trích xuất,13833
Nội dung rỗng,129
Bình luận hợp lệ,13704


## 4. Làm sạch và định danh bình luận

Chuẩn hóa Unicode, HTML, URL, số điện thoại, dấu câu và khoảng trắng; khử trùng
và loại nội dung không có chữ/số. Giữ nguyên `text` và bổ sung `cleaned_text`.
Mã bình luận được liên kết theo nội dung và metadata để khớp gợi ý gán nhãn.


In [4]:
cleaned_path = INTERMEDIATE_DIR / "comments_cleaned.csv"
clean_facebook_comments([comments_path], cleaned_path)
cleaned = pd.read_csv(cleaned_path, dtype=str, keep_default_na=False)
annotation_index = pd.read_csv(inputs["annotation_index"], dtype=str, keep_default_na=False)
predictions = json.loads(inputs["predictions"].read_text(encoding="utf-8"))
cleaned, linked_predictions = attach_annotation_ids(cleaned, annotation_index, predictions)
cleaned.to_csv(cleaned_path, index=False, encoding="utf-8-sig")
display(cleaned.loc[cleaned.text.ne(cleaned.cleaned_text), ["text", "cleaned_text"]].head(8))
display(pd.Series({"Trước xử lý": len(comments), "Sau xử lý": len(cleaned)}).to_frame("Số bình luận"))
show_code("pipeline/prepare/clean_for_labeling.py", "preprocess")


  - File: comments.csv           | thô:  13704 | giữ:  11366
  - Tổng dòng thô:             13704
  - Bỏ (dòng trùng lặp):        2338
  - Bỏ (rỗng/chỉ dấu câu):       392
  - Còn lại sau làm sạch:      10974 (80.1%)
  - Có chứa URL:                 331
  - Có chứa SĐT:                 109
  -> Đã lưu CSV: C:\KH086\PT&KP DLVB\PROJECT\ptit-facebook-spam-ham\data\intermediate\comments_cleaned.csv


#,text,cleaned_text
12,"IntelligentOctopus8657 AI nó về bài toán thực tế, nếu đề tài hay làm dạng nghiên cứu đưa ra giải pháp thì điểm đã cao rồi còn đề tài chung chung xử lý những thứ bình thường như chatbot thì cần tech cao, phải tối ưu hơn những cái đã có","IntelligentOctopus8657 AI nó về bài toán thực tế, nếu đề tài hay làm dạng nghiên cứu đưa ra giải pháp thì điểm đã cao rồi còn đề tài chung chung xử lý những thứ bình thường như chatbot thì cần tech cao, phải tối ưu hơn những cái đã có"
19,"RelaxingAvocado9388 tuỳ vào hội đồng nhé, ví dụ hội đồng nào có thầy hoá là y rằng thầy hỏi sâu về database","RelaxingAvocado9388 tuỳ vào hội đồng nhé, ví dụ hội đồng nào có thầy hoá là y rằng thầy hỏi sâu về database"
36,"FriendlyTurkey7620 a/chi ơi cho em hỏi, thực tập ở vnpt như nào vậy ạ","FriendlyTurkey7620 a chi ơi cho em hỏi, thực tập ở vnpt như nào vậy ạ"
41,"FriendlyTurkey7620 Anh ơi cho em hỏi với, điểm thực tập ở VNPT có được A/A+ không anh","FriendlyTurkey7620 Anh ơi cho em hỏi với, điểm thực tập ở VNPT có được A A không anh"
44,Nguyễn Thành Nam kín nhá,Nguyễn Thành Nam kín nhá
50,"e nên tìm hiểu về học phí nữa, hiện tại tuyển sinh ngành kỹ thuật đang hơn 1 triệu/tín chỉ. Nhưng quan trọng hơn là học phí năm nào cũng tăng khoảng 100k/ tín chỉ. 😅 Lúc a mới vô trường chưa tới 600k/tín mà giờ tăng gần gấp đôi rồi.","e nên tìm hiểu về học phí nữa, hiện tại tuyển sinh ngành kỹ thuật đang hơn 1 triệu tín chỉ. Nhưng quan trọng hơn là học phí năm nào cũng tăng khoảng 100k tín chỉ. Lúc a mới vô trường chưa tới 600k tín mà giờ tăng gần gấp đôi rồi."
52,Nếu phân vân về độ khó thì k nên học mấy cái này 🐧 Còn mình thấy viễn thông thì có nhiều hướng đi hơn,Nếu phân vân về độ khó thì k nên học mấy cái này Còn mình thấy viễn thông thì có nhiều hướng đi hơn
56,ElegantCorgi5380 sao em nghe a chị bảo là kì đông . cả kể ít cũng mở nhưng sẽ * 1.5 học phí,ElegantCorgi5380 sao em nghe a chị bảo là kì đông . cả kể ít cũng mở nhưng sẽ 1.5 học phí


#,Số bình luận
Trước xử lý,13704
Sau xử lý,10974


## 5. Gán nhãn Spam/Ham

Gợi ý nhãn được tạo bằng `ai_assisted_spam_ham.py`, gồm `label` và `confidence`.
Áp dụng ngưỡng tiếp nhận 0,75 bằng `auto_accept_labels.py`; những bình luận chưa
đạt điều kiện không được đưa vào tập có nhãn. Trong chế độ tái lập, gợi ý được
đọc từ cache và không phát sinh lời gọi LLM/Argilla.

Hai nhãn sử dụng: **ham** — bình luận thông thường; **spam** — nội dung rác hoặc
chào mời. Confidence là điểm gợi ý của mô hình, không phải độ chính xác đã đo.


In [5]:
prediction_path = INTERMEDIATE_DIR / "annotation_predictions.json"
prediction_path.write_text(json.dumps(linked_predictions, ensure_ascii=False, indent=2), encoding="utf-8")
review_path = INTERMEDIATE_DIR / "annotation_review.csv"
command = [sys.executable, "-m", "pipeline.label.auto_accept_labels",
           "--csv-in", str(cleaned_path), "--cache", str(prediction_path),
           "--threshold", "0.75", "--skip-argilla", "--export-csv", str(review_path)]
completed = subprocess.run(command, cwd=PROJECT_ROOT, capture_output=True, text=True, encoding="utf-8")
if completed.returncode:
    raise RuntimeError(completed.stderr or completed.stdout)
review = pd.read_csv(review_path, dtype=str, keep_default_na=False)
labeled = review[review.official_label.isin(["ham", "spam"])].copy()
labeled = labeled.rename(columns={"official_label": "label", "ai_confidence": "confidence"})
examples = pd.concat([labeled[labeled.label.eq(label)].head(4) for label in ["ham", "spam"]])
display(examples[["id", "text", "cleaned_text", "label", "confidence"]])
display(pd.Series({"Bình luận sau làm sạch": len(cleaned), "Gợi ý có thể áp dụng": len(linked_predictions),
                   "Bình luận được tiếp nhận": len(labeled)}).to_frame("Số lượng"))
show_code("pipeline/label/ai_assisted_spam_ham.py", "predict_batch_with_glm", 25)


#,id,text,cleaned_text,label,confidence
0,cmt_0,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,Huy Hoàng Nguyễn Hữu Thắng người thầy tôi yêu,ham,0.95
1,cmt_1,"Cô Chi nhé, bao điểm cao luôn","Cô Chi nhé, bao điểm cao luôn",ham,0.95
2,cmt_2,Thầy hùng nhé,Thầy hùng nhé,ham,0.95
3,cmt_3,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,Con_cá_cắn_cáp thầy Hùng bên cntt hay thầy Phạm Hùng bên viễn thông ạ,ham,0.95
162,cmt_159,M cần hoa tốt nghiệp ib cho shop nha Bloom Bite,M cần hoa tốt nghiệp ib cho shop nha Bloom Bite,spam,0.9
280,cmt_276,"các e D22 app VMO đi nhé, chỉ onboard t2 đầu tuần và auto điểm 8 trở lên =))","các e D22 app VMO đi nhé, chỉ onboard t2 đầu tuần và auto điểm 8 trở lên ))",spam,0.92
468,cmt_451,[ THANH XUÂN] tuyển dụng 1. Vị trí: Nhân viên kỹ thuật lắp đặt camera và Sinh viên làm thêm Thu nhập tb : 10.000.000 - 20.000.000 vnđ Không có kinh nghiệm sẽ được đào tạo bài bản. Thời gian: 8h – 17h30 ( Nghỉ trưa 12h – 13h30 ) T2 - T7 * Địa điểm: 30 ngõ 116 Nguyễn Xiển - Hạ Đình - Thanh Xuân - Hà Nội * Quan tâm ib trực tiếp hoặc gửi CV qua mail: cuc.matden@gmail.com Sdt/Zalo: 0985601191,THANH XUÂN tuyển dụng 1. Vị trí: Nhân viên kỹ thuật lắp đặt camera và Sinh viên làm thêm Thu nhập tb : 10.000.000 - 20.000.000 vnđ Không có kinh nghiệm sẽ được đào tạo bài bản. Thời gian: 8h 17h30 ( Nghỉ trưa 12h 13h30 ) T2 - T7 Địa điểm: 30 ngõ 116 Nguyễn Xiển - Hạ Đình - Thanh Xuân - Hà Nội Quan tâm ib trực tiếp hoặc gửi CV qua mail: cuc.matden gmail.com Sdt Zalo: phonetoken,spam,0.98
469,cmt_452,mng quan tâm job thì gửi CV qua zalo 0965987629 nhaaaa🥰🥰,mng quan tâm job thì gửi CV qua zalo phonetoken nhaaaa,spam,0.98


#,Số lượng
Bình luận sau làm sạch,10974
Gợi ý có thể áp dụng,6571
Bình luận được tiếp nhận,6289


## 6. Xuất tập dữ liệu

Lưu các bình luận có nhãn vào `data/labeled/comments_labeled.csv`. Tập dữ liệu giữ ID, nội dung
gốc, nội dung làm sạch, metadata và nhãn để phục vụ kiểm tra, phân tích và huấn luyện.


In [6]:
columns = ["id", "post_id", "group_id", "group_name", "created_at", "comment_type",
           "text", "cleaned_text", "reaction_count", "label", "confidence"]
dataset_path = LABELED_DIR / "comments_labeled.csv"
labeled[columns].to_csv(dataset_path, index=False, encoding="utf-8-sig")
dataset = pd.read_csv(dataset_path, dtype=str, keep_default_na=False)
assert dataset.id.is_unique and dataset.label.isin(["ham", "spam"]).all()
assert dataset[columns].equals(labeled[columns].reset_index(drop=True).astype(str))
distribution = dataset.label.value_counts().rename("Số bình luận").to_frame()
distribution["Tỷ lệ (%)"] = (distribution["Số bình luận"] / len(dataset) * 100).round(2)
display(distribution)
display(pd.DataFrame([file_record(dataset_path, len(dataset), ", ".join(columns), PROJECT_ROOT)]))
manifest = {"mode": "replay_collection_and_cached_annotations", "facebook_requests": False,
            "llm_argilla_calls": False, "posts": len(posts), "groups": len(group_table),
            "raw_rows": len(raw_rows), "nonempty_rows": len(comments), "cleaned_rows": len(cleaned),
            "linked_prediction_count": len(linked_predictions), "labeled_rows": len(dataset),
            "labels": dataset.label.value_counts().to_dict(), "final_csv": str(dataset_path),
            "source_files": {key: str(path) for key, path in inputs.items()}}
(DATA_DIR / "collection_manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")


label,Số bình luận,Tỷ lệ (%)
ham,5190,82.53
spam,1099,17.47


#,file,trạng thái,số bản ghi,KB,chứa
0,data/labeled/comments_labeled.csv,vừa ghi,6289,1633.7,"id, post_id, group_id, group_name, created_at, comment_type, text, cleaned_text, reaction_count, label, confidence"


819

## Nhận xét

Dữ liệu gồm bình luận và phản hồi từ các nhóm PTIT. Quá trình chuẩn hóa giữ
metadata, bảo toàn nội dung gốc và loại các bản ghi không phù hợp. Tỷ lệ ham/spam
cho thấy cần xem xét mất cân bằng lớp khi xây dựng mô hình; chất lượng nhãn là
yếu tố quan trọng khi diễn giải kết quả phân loại.


## Tài liệu tham khảo

- Công cụ thu thập tham khảo: https://github.com/mohdtalal3/facebook_post_comment_scraper
- Các module thu thập: `pipeline/collect/`, `external/facebook_scraper/`.
- Các module chuẩn hóa và gán nhãn: `pipeline/prepare/`, `pipeline/label/`.
